# Monitoring with Jaeger

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pecca-core/pecca/blob/main/examples/notebooks/monitoring_jaeger.ipynb)

Pecca emits one OpenTelemetry span per call (`pecca.call`). Here a local **Jaeger** collects them and we read them back through Jaeger's API.
> Needs **Docker** for the container cells; without it (`PECCA_SKIP_DOCKER=1`) those cells are skipped and a fallback runs.

In [1]:
%pip install -q pecca requests

In [2]:
import json, os, re, shutil, subprocess, tempfile, time
from pathlib import Path

import pandas as pd
import pecca
from pecca.core import context
from pecca.data.dataset import Dataset, canonicalize
from pecca.data.synthetic import generate

HERE = Path.cwd()                                   # the notebook's folder (examples/notebooks in the repo)
work = tempfile.mkdtemp()
os.chdir(work)
os.environ["PECCA_HOME"] = os.path.join(work, ".pecca")
os.environ["TQDM_DISABLE"] = "1"
os.environ["MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT"] = "1"
import logging
try:
    import mlflow  # MLflow resets its logger level on import, so quiet it afterwards
    logging.getLogger("mlflow").setLevel(logging.ERROR)
except ImportError:
    pass

# Docker-backed cells run only when Docker is reachable and PECCA_SKIP_DOCKER != 1 (CI sets it).
USE_DOCKER = (os.environ.get("PECCA_SKIP_DOCKER") != "1" and shutil.which("docker") is not None
              and subprocess.run(["docker", "info"], capture_output=True).returncode == 0)
print("docker:", "available" if USE_DOCKER else "skipped (PECCA_SKIP_DOCKER=1 or Docker not reachable)")

def sh(*args, check=True):
    return subprocess.run([str(a) for a in args], capture_output=True, text=True, check=check)

import atexit
def cleanup_later(*cmd):
    """Run `cmd` when the kernel exits, so containers are removed even if a cell fails."""
    atexit.register(lambda: subprocess.run([str(c) for c in cmd], capture_output=True))

def free_port():
    import socket
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

def wait_for(check, what, seconds=90):
    end = time.time() + seconds
    while time.time() < end:
        try:
            if check():
                return
        except Exception:
            pass
        time.sleep(1)
    raise TimeoutError(f"{what} did not become ready in {seconds}s")

def train_demo(call="route_rfi", rows=3000, candidates=("tfidf_linear",), **kw):
    """Train a small model on the synthetic demo data (LLM answers + human corrections)."""
    df = generate(rows)
    df["text"] = df.email_subject + "\n\n" + df.email_body
    cols = {"input": "text", "llm_output": "llm_rfi", "human_label": "human_rfi", "call_name": {"literal": call.split("/")[-1]}}
    return pecca.train(call, Dataset(canonicalize(df, cols), cols), candidates=list(candidates), **kw)

docker: available


## 1. Start Jaeger
`examples/monitoring/jaeger/docker-compose.yml` (UI on http://localhost:16686; OTLP/HTTP on 4318).

In [3]:
COMPOSE = ["docker", "compose", "-p", "pecca-jaeger", "-f", str(HERE.parent / "monitoring" / "jaeger" / "docker-compose.yml")]
if USE_DOCKER:
    import requests
    cleanup_later(*COMPOSE, "down", "-v")
    sh(*COMPOSE, "up", "-d")
    wait_for(lambda: requests.get("http://127.0.0.1:16686/", timeout=2).status_code == 200, "Jaeger UI")
    print("Jaeger is up")

Jaeger is up


## 2. Point Pecca at it and run some shadow calls
The only Pecca config is `workspace.telemetry`; there is no vendor-specific code.

In [4]:
telemetry = "  telemetry: {exporter: otlp, endpoint: http://127.0.0.1:4318/v1/traces, service: pecca-demo}\n" if USE_DOCKER else ""
Path("pecca.yaml").write_text("version: 1\nworkspace:\n  name: default\n" + telemetry + "projects:\n  default: {governance: {extends: none}}\n")
context.reset()
print(Path("pecca.yaml").read_text())

version: 1
workspace:
  name: default
  telemetry: {exporter: otlp, endpoint: http://127.0.0.1:4318/v1/traces, service: pecca-demo}
projects:
  default: {governance: {extends: none}}



In [5]:
result = train_demo()
print(f"trained {result.winner}: macro_f1 {result.metric:.2f}, threshold {result.threshold:.2f}")

trained tfidf_linear: macro_f1 0.92, threshold 0.59


In [6]:
# The "LLM": replay the simulated LLM's answers for a few new emails (no API needed).
new = generate(30, seed=11)
texts = (new.email_subject + "\n\n" + new.email_body).tolist()
said = dict(zip(texts, new.llm_rfi))

@pecca.replace("route_rfi", mode="shadow")
def route_rfi(text: str) -> str:
    return said[text]                      # stand-in for my_llm.classify(text), unchanged

pecca.promote("route_rfi", "shadow")
for t in texts:
    route_rfi(t)

from opentelemetry import trace
trace.get_tracer_provider().force_flush() if hasattr(trace.get_tracer_provider(), "force_flush") else None
from pecca.runtime.logging_sink import flush_all
flush_all()

print(f"ran {len(texts)} shadow calls")

ran 30 shadow calls


## 3. Read the spans back from Jaeger

In [7]:
if USE_DOCKER:
    def fetch():
        r = requests.get("http://127.0.0.1:16686/api/traces", params={"service": "pecca-demo", "limit": 100}, timeout=10).json()
        return r["data"]
    wait_for(lambda: len(fetch()) >= len(texts), "spans in Jaeger", 60)
    rows = []
    for tr in fetch():
        for sp in tr["spans"]:
            if sp["operationName"] == "pecca.call":
                tags = {t["key"]: t["value"] for t in sp["tags"]}
                rows.append({"mode": tags.get("pecca.mode"), "served_by": tags.get("pecca.served_by"), "version": tags.get("pecca.version"),
                             "confidence": round(tags.get("pecca.confidence", float("nan")), 2), "agreement": tags.get("pecca.agreement"),
                             "latency_ms": round(tags.get("pecca.latency_ms", float("nan")), 2)})
    spans = pd.DataFrame(rows)
    print(f"{len(spans)} pecca.call spans received by Jaeger")
    display(spans.head(8))
    print(spans.groupby(["mode", "served_by"]).size().rename("spans").to_frame())
    print(f"agreement with the LLM: {spans['agreement'].mean():.0%}")
else:
    print("skipped: needs Docker. With Docker, this cell lists the pecca.call spans Jaeger received.")

30 pecca.call spans received by Jaeger


,mode,served_by,version,confidence,agreement,latency_ms
0,shadow,llm,v1,0.97,True,0.52
1,shadow,llm,v1,0.93,True,0.72
2,shadow,llm,v1,1.00,True,0.61
3,shadow,llm,v1,0.90,True,0.49
4,shadow,llm,v1,0.95,True,0.56
5,shadow,llm,v1,0.93,True,0.52
6,shadow,llm,v1,0.95,True,0.53
7,shadow,llm,v1,0.98,False,0.58


                  spans
mode   served_by       
shadow llm           30
agreement with the LLM: 80%


## 4. Clean up

In [8]:
if USE_DOCKER:
    sh(*COMPOSE, "down", "-v")
    print("Jaeger stopped")

Jaeger stopped
